In [ ]:
# ── WIDGETS ──────────────────────────────────────────────────
dbutils.widgets.text("catalog_param", "my_assessment")
catalog = dbutils.widgets.get("catalog_param")

In [ ]:
# ── 1. OPTIMIZE WITH Z-ORDER ──────────────────────────────────
# Z-ORDER clusters data by a column physically on disk
# Makes range queries and filters on that column MUCH faster

# Optimize fact_sales by order_date (most queried column)
spark.sql(f"""
    OPTIMIZE {catalog}.gold.fact_sales
    ZORDER BY (order_date)
""")
print("✅ Z-ORDER on fact_sales done")

# Optimize orders by customer_id (frequently joined column)
spark.sql(f"""
    OPTIMIZE {catalog}.silver.orders_cleaned
    ZORDER BY (customer_id)
""")
print("✅ Z-ORDER on orders_cleaned done")

In [ ]:
# ── 2. LIQUID CLUSTERING ──────────────────────────────────────
# Modern alternative to Z-ORDER
# More flexible, no need to re-cluster when query patterns change

spark.sql(f"""
    ALTER TABLE {catalog}.gold.revenue_by_state
    CLUSTER BY (state)
""")

spark.sql(f"OPTIMIZE {catalog}.gold.revenue_by_state")
print("✅ Liquid clustering on revenue_by_state done")

In [ ]:
# ── 3. VACUUM ─────────────────────────────────────────────────
# Removes old files that are no longer needed
# Keeps storage clean and costs low
# RETAIN 168 HOURS = keep last 7 days of history

spark.sql(f"VACUUM {catalog}.gold.fact_sales RETAIN 168 HOURS")
spark.sql(f"VACUUM {catalog}.silver.orders_cleaned RETAIN 168 HOURS")
spark.sql(f"VACUUM {catalog}.gold.revenue_by_state RETAIN 168 HOURS")
print("✅ VACUUM done")

In [ ]:
# ── 4. TIME TRAVEL ────────────────────────────────────────────
# Delta keeps history of all changes
# You can query older versions of a table

# See full history
spark.sql(f"DESCRIBE HISTORY {catalog}.gold.fact_sales").display()

In [ ]:
# Query version 0 (the very first version)
spark.sql(f"""
    SELECT * FROM {catalog}.gold.fact_sales VERSION AS OF 0
""").display()
print("✅ Time Travel done")

In [ ]:
# ── 5. SCHEMA ENFORCEMENT ─────────────────────────────────────
# Delta rejects data that doesn't match the table schema
# Protects data quality automatically

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Try writing wrong schema → Delta will REJECT it
wrong_schema_df = spark.createDataFrame(
    [("bad_data", 999)],
    ["wrong_col_1", "wrong_col_2"]
)

try:
    wrong_schema_df.write.mode("append") \
        .saveAsTable(f"{catalog}.gold.fact_sales")
except Exception as e:
    print(f"✅ Schema enforcement working — Delta rejected bad data:")
    print(f"   {str(e)[:100]}")

In [ ]:
# ── 6. PARTITION PRUNING VALIDATION ───────────────────────────
# Prove that partitioning actually speeds up queries
# Compare full scan vs partition pruned scan

from pyspark.sql.functions import col
import time

# Full scan — reads ALL partitions
start = time.time()
spark.table(f"{catalog}.gold.fact_sales").count()
full_scan_time = time.time() - start
print(f"Full scan time: {full_scan_time:.2f}s")

# Pruned scan — reads ONLY matching partitions
start = time.time()
spark.table(f"{catalog}.gold.fact_sales") \
    .filter(col("order_date") >= "2024-01-01") \
    .count()
pruned_time = time.time() - start
print(f"Pruned scan time: {pruned_time:.2f}s")

print(f"✅ Partition pruning is {full_scan_time/pruned_time:.1f}x faster")